# Procedural Frozen Lake with explicit Q* metadata

This notebook shows one custom environment and one environment tool together: `Procedural-FrozenLake-v1` plus an explicit `q_star_source`. The wrapper stack adds exact Q-values to `result[i]["q_star"]` each step, so a rollout can choose greedy actions without a separate policy network.

## Imports

Discrete actions are passed as `TensorDict` with an `action.discrete` field (one integer per env).

In [ ]:
import numpy as np
import torch
from tensordict import TensorDict

from mouse_envs import EnvConfig, make_vector_env

## Environment with explicit Q* source

In [ ]:
cfg = EnvConfig(
    group_id="Procedural-FrozenLake-v1",
    seed=7,
    num_envs=1,
    max_episode_steps=200,
    q_star_source={"provider": "metadata_q_star"},
)
env = make_vector_env(cfg)

## Inspect the reset frame

The first `step` performs the internal reset and already includes `q_star` for the start state.

In [ ]:
result, metrics = env.step(env.sample_random_actions())

print(f"observation: {result[0]['observation']['discrete'].tolist()}")
print(f"q_star:      {result[0]['q_star']}")

## Expert rollout

Take `argmax` over Q* to choose greedy actions, then feed those actions back into the same step-only API.

In [ ]:
episodes = 0

for step in range(200):
    actions = [
        TensorDict(
            {
                "action": {
                    "discrete": torch.tensor(
                        [int(np.argmax(result[i]["q_star"]))],
                        dtype=torch.int64,
                    )
                }
            },
            batch_size=[],
        )
        for i in range(env.num_envs)
    ]
    result, metrics = env.step(actions)

    r = result[0]
    if r["done"].item() != 0:
        episodes += 1
        done_code = r["done"].item()
        outcome = "terminated" if done_code == 1 else "truncated"
        print(
            f"step={step:3d}  episode={episodes}  outcome={outcome}  "
            f"reward={r['reward'].item():.1f}"
        )

## Cleanup

In [ ]:
env.close()